# Stage 3: Building the retrieval part of my RAG system

**Project:** Agentic AI system for automated research and report generation  
**Application topic:** AI in healthcare  
**Student:** Adarsh Konderu

In Stage 2, I created a corpus from the PubMed Central Open Access collection.
After title and abstract screening, 46 CC BY articles were retained and four
were excluded with recorded reasons.

This notebook now completes the **retrieval** part of Retrieval-Augmented
Generation (RAG). It:

1. loads the final 46-article corpus;
2. divides long articles into overlapping text chunks;
3. converts the chunks into numerical embeddings;
4. retrieves the most relevant evidence for a research question; and
5. saves the data and test results for reproducibility.

This stage does not generate a report and does not make clinical decisions.
The corpus contains public research articles, not patient records.


## 1. Why I am implementing retrieval directly

I am using plain Python and Sentence Transformers rather than LangChain at this
stage. This makes the main RAG operations visible and easier for me to explain:

`question → question embedding → similarity comparison → relevant chunks`

The model used here creates embeddings only. It is not an LLM judge and it does
not write the final answer.


In [ ]:
# Sentence Transformers provides the embedding model used for semantic search.
# The command is kept in its own cell so the installed package is easy to identify.
!pip -q install sentence-transformers


## 2. Import the libraries and record my settings

The chunk size and overlap are fixed before running the experiment. Keeping
these values fixed makes later comparisons reproducible.


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import re
import shutil
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from google.colab import files
from sentence_transformers import SentenceTransformer

# All Stage 3 evidence files will be stored in one folder.
OUTPUT_DIR = Path("/content/stage_3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed experimental settings.
EXPECTED_ARTICLES = 46
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MINIMUM_FINAL_CHUNK_WORDS = 80
TOP_K = 5

print("Embedding model:", MODEL_NAME)
print("Chunk size:", CHUNK_SIZE_WORDS, "words")
print("Chunk overlap:", CHUNK_OVERLAP_WORDS, "words")


## 3. Upload and validate the final corpus

Upload `pmc_corpus_final.jsonl` from the Stage 3 package. The checks below stop
the notebook if the file does not contain the expected 46 unique PMC articles.


In [ ]:
# Colab displays an upload button when this cell runs.
uploaded_files = files.upload()

expected_name = "pmc_corpus_final.jsonl"
if expected_name in uploaded_files:
    uploaded_name = expected_name
elif len(uploaded_files) == 1:
    # This still works if the browser slightly changed the downloaded filename.
    uploaded_name = next(iter(uploaded_files))
else:
    raise ValueError("Please upload only pmc_corpus_final.jsonl and run this cell again.")

corpus_bytes = uploaded_files[uploaded_name]
corpus_path = Path("/content/pmc_corpus_final.jsonl")
corpus_path.write_bytes(corpus_bytes)

# JSONL stores one complete article record on each line.
records = [
    json.loads(line)
    for line in corpus_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

pmcids = [record["pmcid"] for record in records]
assert len(records) == EXPECTED_ARTICLES, f"Expected 46 articles, found {len(records)}"
assert len(set(pmcids)) == EXPECTED_ARTICLES, "A duplicate PMCID was found"
assert all(record["licence"] == "CC BY" for record in records), "Non-CC BY record found"
assert all(record.get("full_text", "").strip() for record in records), "Missing full text"

print("Corpus loaded correctly")
print("Articles:", len(records))
print("Themes:", dict(Counter(record["theme"] for record in records)))


## 4. Split each article into overlapping chunks

A complete article is too large and too general to retrieve as a single item.
I therefore use 220-word chunks with a 40-word overlap. The overlap reduces the
chance of losing an important idea at the boundary between two chunks.

The final part of an article is joined to the preceding chunk when it would be
shorter than 80 words. This avoids very small, low-context chunks.


In [ ]:
def clean_text(text):
    """Remove repeated whitespace while keeping the article wording unchanged."""
    return re.sub(r"\s+", " ", text).strip()


def split_into_word_chunks(text, chunk_size, overlap, minimum_final_size):
    """Split text into fixed word windows with a controlled overlap."""
    words = clean_text(text).split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        # Absorb a short remainder so no article ends with a tiny chunk.
        if 0 < len(words) - end < minimum_final_size:
            end = len(words)

        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break
        start = end - overlap

    return chunks


def build_chunks(article_records):
    """Create traceable chunks while retaining citation metadata."""
    chunk_records = []

    for article in article_records:
        article_chunks = split_into_word_chunks(
            article["full_text"],
            CHUNK_SIZE_WORDS,
            CHUNK_OVERLAP_WORDS,
            MINIMUM_FINAL_CHUNK_WORDS,
        )

        for position, chunk_text in enumerate(article_chunks, start=1):
            chunk_records.append(
                {
                    "chunk_id": f'{article["pmcid"]}_C{position:04d}',
                    "pmcid": article["pmcid"],
                    "doi": article["doi"],
                    "title": article["title"],
                    "authors": article["authors"],
                    "year": article["year"],
                    "theme": article["theme"],
                    "source_url": article["source_url"],
                    "chunk_position": position,
                    "chunk_word_count": len(chunk_text.split()),
                    "text": chunk_text,
                }
            )

    return chunk_records


In [ ]:
chunks = build_chunks(records)

# Integrity checks connect every chunk back to one of the 46 articles.
chunk_ids = [chunk["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk ID found"
assert {chunk["pmcid"] for chunk in chunks} == set(pmcids), "Article lost during chunking"
assert all(chunk["chunk_word_count"] >= MINIMUM_FINAL_CHUNK_WORDS for chunk in chunks)

chunk_counts = Counter(chunk["pmcid"] for chunk in chunks)
print("Chunks created:", len(chunks))
print("Articles represented:", len(chunk_counts))
print("Minimum chunks from one article:", min(chunk_counts.values()))
print("Maximum chunks from one article:", max(chunk_counts.values()))


## 5. Create semantic embeddings

An embedding is a numerical representation of meaning. Similar passages should
have vectors pointing in similar directions. I include the article title and
theme with the chunk text because they provide useful context for retrieval.

The embeddings are normalised, which means a dot product can be used as cosine
similarity during search.


In [ ]:
# Download the public embedding model and load it into memory.
embedding_model = SentenceTransformer(MODEL_NAME)

texts_for_embedding = [
    f'Title: {chunk["title"]}\nTheme: {chunk["theme"]}\nText: {chunk["text"]}'
    for chunk in chunks
]

embeddings = embedding_model.encode(
    texts_for_embedding,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

assert embeddings.shape[0] == len(chunks)
assert np.isfinite(embeddings).all(), "The embedding matrix contains invalid values"

print("Embedding matrix shape:", embeddings.shape)


## 6. Retrieve evidence for a question

The function below embeds a question and compares it with every chunk. It
returns the highest-scoring evidence. I limit results to two chunks from one
article so a long paper does not dominate the retrieved evidence.


In [ ]:
def search_corpus(question, top_k=TOP_K, maximum_per_article=2):
    """Return the most semantically similar evidence chunks for a question."""
    if not question.strip():
        raise ValueError("The research question cannot be empty")

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    # Normalised vectors allow cosine similarity through a dot product.
    similarity_scores = embeddings @ question_embedding
    ranked_indexes = np.argsort(-similarity_scores)

    selected_results = []
    selected_per_article = Counter()

    for chunk_index in ranked_indexes:
        chunk = chunks[int(chunk_index)]
        if selected_per_article[chunk["pmcid"]] >= maximum_per_article:
            continue

        result = dict(chunk)
        result["similarity_score"] = round(float(similarity_scores[chunk_index]), 6)
        selected_results.append(result)
        selected_per_article[chunk["pmcid"]] += 1

        if len(selected_results) == top_k:
            break

    return selected_results


## 7. Run a small automated retrieval sanity check

These five questions cover the five corpus themes. `Theme match at 5` checks
how many of the first five results come from an expected theme. This is a
development sanity check, not the final dissertation evaluation.


In [ ]:
TEST_QUERIES = [
    {
        "query_id": "Q1",
        "question": "How is artificial intelligence used to support medical imaging diagnosis?",
        "expected_themes": ["medical_imaging_and_diagnosis"],
    },
    {
        "query_id": "Q2",
        "question": "How does AI support clinical decisions and treatment recommendations?",
        "expected_themes": ["clinical_decision_support"],
    },
    {
        "query_id": "Q3",
        "question": "How can artificial intelligence improve healthcare operations and clinical workflows?",
        "expected_themes": ["healthcare_operations"],
    },
    {
        "query_id": "Q4",
        "question": "What hallucination and factual accuracy risks occur when healthcare reports use large language models?",
        "expected_themes": ["generative_ai_and_llms", "ethics_safety_and_bias"],
    },
    {
        "query_id": "Q5",
        "question": "What ethical, safety and bias limitations are reported for AI in healthcare?",
        "expected_themes": ["ethics_safety_and_bias"],
    },
]

retrieval_rows = []
sanity_rows = []

for test_query in TEST_QUERIES:
    results = search_corpus(test_query["question"])
    expected = set(test_query["expected_themes"])
    matching_results = sum(result["theme"] in expected for result in results)

    sanity_rows.append(
        {
            "query_id": test_query["query_id"],
            "question": test_query["question"],
            "expected_themes": " | ".join(test_query["expected_themes"]),
            "theme_match_at_5": matching_results / len(results),
        }
    )

    for rank, result in enumerate(results, start=1):
        retrieval_rows.append(
            {
                "query_id": test_query["query_id"],
                "question": test_query["question"],
                "rank": rank,
                "similarity_score": result["similarity_score"],
                "chunk_id": result["chunk_id"],
                "pmcid": result["pmcid"],
                "theme": result["theme"],
                "title": result["title"],
                "source_url": result["source_url"],
                "text_preview": result["text"][:500],
            }
        )

for sanity_result in sanity_rows:
    print(
        sanity_result["query_id"],
        "theme match at 5 =",
        f'{sanity_result["theme_match_at_5"]:.2f}',
    )


## 8. Inspect one result in a readable form

This display helps me check that the retrieved passage genuinely relates to the
question and still carries its PMCID, title and source URL.


In [ ]:
example_question = TEST_QUERIES[3]["question"]
example_results = search_corpus(example_question, top_k=3)

print("Question:", example_question)
for rank, result in enumerate(example_results, start=1):
    print("\n", "-" * 80)
    print("Rank:", rank, "Score:", result["similarity_score"])
    print("PMCID:", result["pmcid"])
    print("Title:", result["title"])
    print("Theme:", result["theme"])
    print("Source:", result["source_url"])
    print("Passage:", result["text"][:700], "...")


## 9. Save reproducible evidence files

The output contains the chunks, chunk manifest, embedding matrix, test results,
settings and checksums. These files allow me to demonstrate what the retriever
used and to repeat the same search later.


In [ ]:
def write_csv(path, rows, fieldnames):
    """Write a list of dictionaries to a UTF-8 CSV file."""
    with path.open("w", encoding="utf-8", newline="") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


# Save full chunks as JSONL and a lighter manifest as CSV.
chunks_path = OUTPUT_DIR / "pmc_chunks.jsonl"
with chunks_path.open("w", encoding="utf-8") as file_handle:
    for chunk in chunks:
        file_handle.write(json.dumps(chunk, ensure_ascii=False) + "\n")

manifest_fields = [
    "chunk_id", "pmcid", "doi", "title", "year", "theme",
    "source_url", "chunk_position", "chunk_word_count",
]
write_csv(OUTPUT_DIR / "pmc_chunk_manifest.csv", chunks, manifest_fields)

np.save(OUTPUT_DIR / "pmc_chunk_embeddings.npy", embeddings)
write_csv(
    OUTPUT_DIR / "retrieval_test_results.csv",
    retrieval_rows,
    list(retrieval_rows[0].keys()),
)
write_csv(
    OUTPUT_DIR / "retrieval_sanity_summary.csv",
    sanity_rows,
    list(sanity_rows[0].keys()),
)

corpus_checksum = hashlib.sha256(corpus_bytes).hexdigest()
run_metadata = {
    "stage": "Stage 3 semantic retrieval prototype",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "sentence_transformers_version": importlib.metadata.version("sentence-transformers"),
    "numpy_version": np.__version__,
    "embedding_model": MODEL_NAME,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "minimum_final_chunk_words": MINIMUM_FINAL_CHUNK_WORDS,
    "article_count": len(records),
    "chunk_count": len(chunks),
    "embedding_dimensions": int(embeddings.shape[1]),
    "top_k": TOP_K,
    "maximum_chunks_per_article": 2,
    "corpus_sha256": corpus_checksum,
    "framework": "Direct Python implementation; no LangChain",
}
(OUTPUT_DIR / "retrieval_run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
)

archive_path = shutil.make_archive(
    "/content/Adarsh_Konderu_Stage_3_Retrieval_Evidence",
    "zip",
    OUTPUT_DIR,
)

print("Saved chunks:", len(chunks))
print("Saved retrieval rows:", len(retrieval_rows))
print("Evidence package:", archive_path)
files.download(archive_path)


## 10. What I can explain from this stage

- I used 46 screened, CC BY research articles from PMC.
- I divided the full text into overlapping chunks because retrieval works more
  precisely on focused passages than on complete articles.
- I used `all-MiniLM-L6-v2` to create semantic embeddings.
- I compared a question vector with the chunk vectors using cosine similarity.
- Every result retains its PMCID, DOI, title and URL for traceable citations.
- I saved the settings, checksum and test outputs so the process can be repeated.

The next development stage will connect these retrieved chunks to the planner,
writer and reviewer agents. The final experiment will compare a single-pass
LLM, an agentic system without RAG and the complete agentic RAG system.
